In [1]:
import sys, os
import pandas as pd
sys.path.append(os.path.abspath('..'))
%load_ext autoreload
%autoreload 2

In [2]:
from src.data_cleaning import load_processed_splits

X_train, X_test, y_train, y_test = load_processed_splits()
X_train.head()

,الطابق,عدد الغرف,عدد الحمامات,المساحة (م2),الوجهة,الإكساء,عدد الشقق في كل طابق,موقف سيارة,وجود مصعد,اتجاه_شرقي,اتجاه_شمالي,اتجاه_غربي,الشارع_encoded
0,3,5,2,140,1,2,2,0,0,0,1,0,103689.422672
1,2,4,1,120,0,1,2,0,1,0,1,0,94772.056852
2,4,4,1,120,0,1,2,0,1,0,0,1,94772.056852
3,0,6,2,180,1,0,2,0,0,0,0,0,94772.056852
4,4,4,1,100,0,2,3,0,0,0,0,1,106203.913748


## Baseline model comparison (5-fold CV)

In [3]:
from src.train import evaluate_models
results_df = evaluate_models(X_train, y_train, cv_folds=5)
results_df

,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std
0,Gradient Boosting,11240.045451,3318.369099,6876.191053,1464.885700,0.874536,0.079570
1,Ridge Regression,11742.495978,2113.573007,9249.845604,1441.748333,0.869456,0.049529
2,Linear Regression,12097.678050,2432.565295,9468.603292,1515.070729,0.860825,0.057662
3,Random Forest,12706.014940,3663.275974,7847.507905,1743.909623,0.840402,0.083627
4,XGBoost,13044.951660,5500.117969,7557.469043,2190.012693,0.829952,0.113898


## Hyperparameter tuning (tree models)

In [4]:
from src.train import tune_tree_models
best_estimators, tuned_results_df = tune_tree_models(X_train, y_train, cv_folds=5)
tuned_results_df

,Model,Best Params,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std
0,XGBoost,"{'learning_rate': 0.1, 'max_depth': 2, 'n_esti...",9025.704199,2707.968954,5766.976270,970.536980,0.915905,0.067150
1,Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 2, 'min_sa...",9853.688842,2858.510085,6409.403145,1127.746875,0.899750,0.070151
2,Random Forest,"{'max_depth': None, 'min_samples_leaf': 3, 'n_...",12406.290353,3154.661048,8154.824057,1494.815407,0.847599,0.092294


## Finalize XGBoost + Ridge, evaluate on test set, compare to average

In [5]:
# Cell 5
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from src.train import evaluate_on_test, log_results

final_xgb = best_estimators['XGBoost']
final_xgb.fit(X_train, y_train)

final_ridge = Ridge(alpha=1.0, random_state=42)
final_ridge.fit(X_train, y_train)

xgb_metrics, xgb_preds = evaluate_on_test(final_xgb, X_test, y_test)
ridge_metrics, ridge_preds = evaluate_on_test(final_ridge, X_test, y_test)

avg_preds = (xgb_preds + ridge_preds) / 2
avg_metrics = {
    'RMSE': mean_squared_error(y_test, avg_preds) ** 0.5,
    'MAE': mean_absolute_error(y_test, avg_preds),
    'R2': r2_score(y_test, avg_preds)
}

log_results({'Model': 'XGBoost (tuned)', 'Stage': 'test_set_final', **xgb_metrics})
log_results({'Model': 'Ridge Regression', 'Stage': 'test_set_final', **ridge_metrics})
log_results({'Model': 'Average (XGBoost + Ridge)', 'Stage': 'test_set_final', **avg_metrics})

print("XGBoost:", xgb_metrics)
print("Ridge:", ridge_metrics)
print("Average:", avg_metrics)

Logged: XGBoost (tuned) | RMSE=6380.72 | MAE=4932.82 | R2=0.9707
Logged: Ridge Regression | RMSE=10811.59 | MAE=9366.24 | R2=0.9159
Logged: Average (XGBoost + Ridge) | RMSE=7514.94 | MAE=6403.68 | R2=0.9594
XGBoost: {'RMSE': 6380.72409684042, 'MAE': 4932.81884765625, 'R2': 0.9707186222076416}
Ridge: {'RMSE': 10811.593160103026, 'MAE': 9366.239341053511, 'R2': 0.9159319370052832}
Average: {'RMSE': 7514.942653181977, 'MAE': 6403.678919799596, 'R2': 0.9593834535920688}


## Feature importance

In [6]:
importances = pd.Series(final_xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)

importance_df = importances.reset_index()
importance_df.columns = ['Feature', 'Importance']
importance_df['Model'] = 'XGBoost (tuned)'
importance_df['timestamp'] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

log_path = "../results/feature_importance_log.csv"
importance_df.to_csv(log_path, mode='a', header=not os.path.exists(log_path), index=False)
importances

عدد الحمامات            0.308152
المساحة (م2)            0.296897
الوجهة                  0.136521
عدد الشقق في كل طابق    0.086952
عدد الغرف               0.062086
الشارع_encoded          0.039980
الإكساء                 0.021699
اتجاه_غربي              0.018692
الطابق                  0.012975
اتجاه_شمالي             0.012436
وجود مصعد               0.001908
اتجاه_شرقي              0.001701
موقف سيارة              0.000000
dtype: float32

## Feature reduction test (drop near-zero importance features)

In [7]:

from sklearn.base import clone
from sklearn.model_selection import cross_validate, KFold

drop_cols = ['موقف سيارة', 'وجود مصعد', 'اتجاه_شرقي']
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'RMSE': 'neg_root_mean_squared_error', 'MAE': 'neg_mean_absolute_error', 'R2': 'r2'}

cv_full = cross_validate(clone(final_xgb), X_train, y_train, cv=kf, scoring=scoring)
cv_reduced = cross_validate(clone(final_xgb), X_train.drop(columns=drop_cols), y_train, cv=kf, scoring=scoring)

log_results({'Model': 'XGBoost (tuned, full features)', 'Stage': 'cv_train_comparison',
             'RMSE': -cv_full['test_RMSE'].mean(), 'MAE': -cv_full['test_MAE'].mean(), 'R2': cv_full['test_R2'].mean()})
log_results({'Model': 'XGBoost (tuned, reduced features)', 'Stage': 'cv_train_comparison',
             'RMSE': -cv_reduced['test_RMSE'].mean(), 'MAE': -cv_reduced['test_MAE'].mean(), 'R2': cv_reduced['test_R2'].mean()})

Logged: XGBoost (tuned, full features) | RMSE=9025.70 | MAE=5766.98 | R2=0.9159
Logged: XGBoost (tuned, reduced features) | RMSE=8974.12 | MAE=5769.46 | R2=0.9174


## Final model: train, evaluate, save

In [8]:
import joblib

X_train_final = X_train.drop(columns=drop_cols)
X_test_final = X_test.drop(columns=drop_cols)

final_model = clone(final_xgb)
final_model.fit(X_train_final, y_train)

final_metrics, final_preds = evaluate_on_test(final_model, X_test_final, y_test)
log_results({'Model': 'XGBoost (FINAL, reduced features)', 'Stage': 'test_set_final', **final_metrics})
print("Final model test performance:", final_metrics)

joblib.dump(final_model, '../models/xgboost_final_model.pkl')
joblib.dump(list(X_train_final.columns), '../models/feature_order.pkl')

# Save test predictions too - needed for the visualization notebook's actual-vs-predicted plot
pd.DataFrame({'y_test': y_test.values, 'y_pred': final_preds}).to_csv('../results/final_test_predictions.csv', index=False)

print("Model artifacts saved.")

Logged: XGBoost (FINAL, reduced features) | RMSE=6285.57 | MAE=4850.71 | R2=0.9716
Final model test performance: {'RMSE': 6285.567595690941, 'MAE': 4850.7060546875, 'R2': 0.9715854525566101}
Model artifacts saved.
